<a href="https://colab.research.google.com/github/iandrukhiv-cell/addressbook-python/blob/main/%D0%90%D0%BD%D0%B4%D1%80%D1%83%D1%85%D1%96%D0%B2_%D0%86%D0%BB%D0%BE%D0%BD%D0%B0_%D0%92%D0%BE%D0%BB%D0%BE%D0%B4%D0%B8%D0%BC%D0%B8%D1%80%D1%96%D0%B2%D0%BD%D0%B0_%D0%9F%D0%A01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ChefBot — кулінарний асистент на базі LangChain

## Крок 1. Підготовка середовища

На цьому етапі встановлюються необхідні бібліотеки для роботи LangChain, OpenAI API, інструментів агента та збереження змінних середовища.

In [7]:
# Встановлення необхідних бібліотек

!pip install -q \
langchain \
langchain-openai \
langchain-community \
python-dotenv \
wikipedia \
numexpr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Крок 2. Імпорт бібліотек та налаштування API

In [8]:

import os
from datetime import datetime
from typing import List, Dict, Any

import numexpr as ne
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper

print("✅ Імпорти для чатбота успішно виконано.")

✅ Імпорти для чатбота успішно виконано.


/tmp/ipykernel_5087/2265562115.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper


## Крок 3. Налаштування API-ключа OpenAI

На цьому етапі налаштовується доступ до OpenAI API через змінну середовища OPENAI_API_KEY. Ключ використовується для роботи мовної моделі ChatOpenAI у LangChain.

In [9]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Введіть OpenAI API Key: ")

print("✅ API-ключ успішно налаштовано")

Введіть OpenAI API Key: ··········
✅ API-ключ успішно налаштовано


##Крок 4. Створення кулінарних інструментів ChefBot

In [16]:
# КРОК 4. Створення кулінарних інструментів ChefBot

from langchain_core.tools import tool

# База рецептів
RECIPES_DB = {
    "борщ": {
        "ingredients": ["буряк", "капуста", "картопля", "морква", "цибуля", "томатна паста"],
        "time_minutes": 90,
        "servings": 6,
        "instructions": "Зваріть бульйон, додайте овочі, томатну пасту та варіть до готовності.",
        "tags": ["перша страва", "українська кухня"]
    },
    "омлет": {
        "ingredients": ["яйця", "молоко", "сіль", "олія"],
        "time_minutes": 10,
        "servings": 2,
        "instructions": "Збийте яйця з молоком, додайте сіль і обсмажте на сковороді.",
        "tags": ["сніданок", "швидко"]
    },
    "курка з картоплею": {
        "ingredients": ["курка", "картопля", "цибуля", "олія", "сіль", "перець"],
        "time_minutes": 60,
        "servings": 4,
        "instructions": "Наріжте картоплю, додайте курку зі спеціями та запікайте до готовності.",
        "tags": ["вечеря", "запікання"]
    },
    "гречка з грибами": {
        "ingredients": ["гречка", "гриби", "цибуля", "олія", "сіль"],
        "time_minutes": 35,
        "servings": 3,
        "instructions": "Відваріть гречку, обсмажте гриби з цибулею та змішайте.",
        "tags": ["гарнір", "вегетаріанське"]
    },
    "салат цезар": {
        "ingredients": ["курка", "салат", "сухарики", "пармезан", "соус"],
        "time_minutes": 25,
        "servings": 2,
        "instructions": "Обсмажте курку, змішайте з салатом, сухариками, сиром і соусом.",
        "tags": ["салат"]
    },
    "млинці": {
        "ingredients": ["борошно", "молоко", "яйця", "цукор", "сіль", "олія"],
        "time_minutes": 30,
        "servings": 4,
        "instructions": "Змішайте інгредієнти в тісто та обсмажте млинці з двох боків.",
        "tags": ["десерт", "сніданок"]
    },
    "рис з овочами": {
        "ingredients": ["рис", "морква", "цибуля", "перець", "олія", "сіль"],
        "time_minutes": 40,
        "servings": 4,
        "instructions": "Відваріть рис, обсмажте овочі та змішайте все разом.",
        "tags": ["гарнір", "вегетаріанське"]
    },
    "сирники": {
        "ingredients": ["творог", "яйця", "борошно", "цукор", "олія"],
        "time_minutes": 25,
        "servings": 3,
        "instructions": "Змішайте інгредієнти, сформуйте сирники та обсмажте.",
        "tags": ["сніданок", "десерт"]
    },
    "овочевий суп": {
        "ingredients": ["картопля", "морква", "цибуля", "капуста", "зелень"],
        "time_minutes": 45,
        "servings": 4,
        "instructions": "Наріжте овочі, залийте водою та варіть до готовності.",
        "tags": ["перша страва", "вегетаріанське"]
    },
    "безглютеновий салат": {
        "ingredients": ["курка", "огірок", "помідор", "салат", "оливкова олія"],
        "time_minutes": 20,
        "servings": 2,
        "instructions": "Наріжте овочі, додайте курку та заправте оливковою олією.",
        "tags": ["без глютену", "салат", "легке"]
    }
}

UNIT_CONVERSIONS = {
    ("склянка", "г", "борошно"): 150,
    ("склянка", "г", "цукор"): 200,
    ("склянка", "г", "рис"): 180,
    ("ст.л.", "мл", ""): 15,
    ("ч.л.", "мл", ""): 5,
    ("ст.л.", "г", "борошно"): 10,
    ("ч.л.", "г", "сіль"): 7,
}

SUBSTITUTIONS = {
    "яйця": [
        "банан — добре підходить для випічки",
        "льняне насіння з водою — веганський варіант",
        "аквафаба — підходить для мусів і меренги"
    ],
    "молоко": [
        "вівсяне молоко",
        "кокосове молоко",
        "мигдальне молоко"
    ],
    "борошно": [
        "рисове борошно",
        "кукурудзяне борошно",
        "мигдальне борошно"
    ],
    "глютен": [
        "рисове борошно",
        "гречане борошно",
        "кукурудзяне борошно"
    ]
}

@tool
def recipe_search(query: str) -> str:
    """
    Шукає рецепти за назвою страви, інгредієнтами або тегами.
    """
    query_lower = query.lower()
    results = []

    for name, recipe in RECIPES_DB.items():
        if query_lower in name.lower():
            results.append((name, recipe, "точний збіг за назвою"))

    if not results:
        query_words = query_lower.replace(",", " ").split()

        for name, recipe in RECIPES_DB.items():
            matched = []

            for ing in recipe["ingredients"]:
                if any(word in ing.lower() for word in query_words):
                    matched.append(ing)

            for tag in recipe["tags"]:
                if any(word in tag.lower() for word in query_words):
                    matched.append(tag)

            if matched:
                results.append((name, recipe, f"збіг: {', '.join(matched)}"))

    if not results:
        return f"Не знайдено рецептів за запитом: {query}"

    output = []

    for name, recipe, match_type in results[:3]:
        output.append(f"🍽️ {name.upper()}")
        output.append(f"Інгредієнти: {', '.join(recipe['ingredients'])}")
        output.append(f"Час: {recipe['time_minutes']} хв | Порцій: {recipe['servings']}")
        output.append(f"Інструкція: {recipe['instructions']}")
        output.append(f"Знайдено за: {match_type}")
        output.append("")

    return "\n".join(output)


@tool
def unit_converter(amount: float, from_unit: str, to_unit: str, product: str = "") -> str:
    """
    Конвертує кулінарні одиниці вимірювання.
    """
    product = product.lower().strip()
    key = (from_unit, to_unit, product)

    if key in UNIT_CONVERSIONS:
        result = amount * UNIT_CONVERSIONS[key]
        return f"{amount} {from_unit} {product} = {result:.1f} {to_unit}"

    key_universal = (from_unit, to_unit, "")

    if key_universal in UNIT_CONVERSIONS:
        result = amount * UNIT_CONVERSIONS[key_universal]
        return f"{amount} {from_unit} = {result:.1f} {to_unit}"

    return f"Не знайдено конвертації з {from_unit} в {to_unit} для продукту: {product or 'не вказано'}"


@tool
def cooking_timer(recipe_name: str, target_servings: int) -> str:
    """
    Розраховує приблизний час приготування залежно від кількості порцій.
    """
    recipe_name = recipe_name.lower().strip()

    if recipe_name not in RECIPES_DB:
        return f"Рецепт '{recipe_name}' не знайдено в базі."

    recipe = RECIPES_DB[recipe_name]
    base_time = recipe["time_minutes"]
    base_servings = recipe["servings"]

    ratio = target_servings / base_servings

    if ratio <= 1:
        new_time = base_time
    else:
        new_time = base_time * (1 + 0.25 * (ratio - 1))

    return (
        f"Для рецепту '{recipe_name}' на {target_servings} порцій "
        f"орієнтовний час приготування становить {round(new_time)} хв. "
        f"Базовий рецепт: {base_time} хв на {base_servings} порцій."
    )


@tool
def substitution_finder(ingredient: str) -> str:
    """
    Шукає замінники інгредієнтів.
    """
    ingredient = ingredient.lower().strip()

    if ingredient in SUBSTITUTIONS:
        variants = SUBSTITUTIONS[ingredient]
        return f"Замінники для '{ingredient}':\n- " + "\n- ".join(variants)

    return f"Замінники для '{ingredient}' не знайдено."


tools = [
    recipe_search,
    unit_converter,
    cooking_timer,
    substitution_finder
]

print("✅ Кулінарні інструменти ChefBot створено.")
print("Інструменти:", [tool.name for tool in tools])

✅ Кулінарні інструменти ChefBot створено.
Інструменти: ['recipe_search', 'unit_converter', 'cooking_timer', 'substitution_finder']


##Крок 5. Ініціалізація мовної моделі (ChatOpenAI) та базова перевірка

In [1]:
!pip install -q langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.2 MB/s eta 0:00:00


In [11]:
# КРОК 5. Ініціалізація чат-моделі Gemini

from langchain_google_genai import ChatGoogleGenerativeAI
import os
from getpass import getpass

# Отримуємо API-ключ
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Введіть Gemini API Key: ")

# Створюємо модель
chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.4,
    max_output_tokens=512,
    timeout=30
)

# Перевіряємо, що модель створена
print("✅ Модель Gemini успішно ініціалізовано.")
print(f"Модель: {chat_model.model}")
print("API-ключ успішно підключено.")

# Тестовий виклик вимкнено через відсутність квоти API
print("⚠️ Тестовий запит до Gemini пропущено через обмеження квоти API.")
print("Крок 5 виконано.")

✅ Модель Gemini успішно ініціалізовано.
Модель: gemini-2.0-flash
API-ключ успішно підключено.
⚠️ Тестовий запит до Gemini пропущено через обмеження квоти API.
Крок 5 виконано.


#Крок 6. Створення агента ChefBot з кулінарними інструментами

In [17]:
# КРОК 6. Створення агента ChefBot з кулінарними інструментами

SYSTEM_PROMPT = """
Ти — ChefBot, дружній кулінарний асистент українською мовою.

Твої можливості:
- шукати рецепти за назвою страви або інгредієнтами;
- конвертувати кулінарні одиниці вимірювання;
- розраховувати час приготування залежно від кількості порцій;
- підбирати замінники інгредієнтів.

Використовуй доступні інструменти:
- recipe_search, коли користувач шукає рецепт або питає, що приготувати;
- unit_converter, коли треба перевести склянки, ложки, грами або мілілітри;
- cooking_timer, коли треба розрахувати час приготування;
- substitution_finder, коли користувач питає, чим замінити інгредієнт.

Відповідай коротко, зрозуміло та українською мовою.
"""

agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Агент ChefBot створений і готовий до тестування.")
print("Підключені інструменти:", [tool.name for tool in tools])

# Тестовий виклик не виконуємо, щоб уникнути помилки квоти Gemini API.
print("⚠️ Тестовий виклик моделі пропущено через можливі обмеження API-квоти.")

✅ Агент ChefBot створений і готовий до тестування.
Підключені інструменти: ['recipe_search', 'unit_converter', 'cooking_timer', 'substitution_finder']
⚠️ Тестовий виклик моделі пропущено через можливі обмеження API-квоти.


Агент ChefBot успішно створено. Тестовий виклик не виконано через обмеження квоти Gemini API, однак структура агента, системний промпт та інструменти підключені коректно.

# Крок 7. Інтерактивний чат із ботом ChefBot та збереження контексту діалогу

In [21]:
# КРОК 7. Інтерактивний чат із ChefBot

def run_chat_session():
    """
    Інтерактивна сесія спілкування з ChefBot.
    Історія діалогу зберігається у messages.
    """

    print("\n" + "=" * 60)
    print("👨‍🍳 ChefBot — кулінарний асистент")
    print("=" * 60)

    print("Можливості:")
    print(" • Пошук рецептів")
    print(" • Конвертація кулінарних мір")
    print(" • Розрахунок часу приготування")
    print(" • Пошук замінників інгредієнтів")
    print(" • Пам'ять контексту розмови")

    print("\nКоманди:")
    print(" • exit / вихід / quit — завершити роботу")
    print(" • /reset — очистити контекст")

    messages = []

    while True:

        user_input = input("\n👤 Ви: ").strip()

        if not user_input:
            continue

        lower_input = user_input.lower()

        if lower_input in ("exit", "вихід", "/exit", "quit"):
            print("\n👋 Роботу завершено.")
            break

        if lower_input == "/reset":
            messages = []
            print("🔄 Контекст очищено.")
            continue

        messages.append(
            {
                "role": "user",
                "content": user_input
            }
        )

        try:

            result_state = agent.invoke(
                {
                    "messages": messages
                }
            )

            messages = result_state["messages"]

            last_message = messages[-1]

            print("\n👨‍🍳 ChefBot:")

            if isinstance(last_message, dict):
                print(last_message.get("content", ""))
            else:
                print(last_message.content)

        except Exception as e:

            print("\n⚠️ Не вдалося отримати відповідь від моделі.")
            print(f"Помилка: {e}")


run_chat_session()


👨‍🍳 ChefBot — кулінарний асистент
Можливості:
 • Пошук рецептів
 • Конвертація кулінарних мір
 • Розрахунок часу приготування
 • Пошук замінників інгредієнтів
 • Пам'ять контексту розмови

Команди:
 • exit / вихід / quit — завершити роботу
 • /reset — очистити контекст

👤 Ви: рецепт борщу

⚠️ Не вдалося отримати відповідь від моделі.
Помилка: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for 

KeyboardInterrupt: Interrupted by user

# Крок 8. Автоматичне тестування типових сценаріїв

In [20]:
# КРОК 8. Автоматичне тестування типових сценаріїв ChefBot

def run_automatic_tests():
    """
    Автоматично перевіряє типові сценарії роботи ChefBot:
    - пошук рецепту;
    - пошук страви за інгредієнтами;
    - конвертацію одиниць;
    - розрахунок часу;
    - пошук замінників;
    - роботу з пам'яттю через messages.
    """

    test_queries = [
        "Що можна приготувати з курки та картоплі?",
        "Знайди рецепт борщу.",
        "Скільки грамів борошна в одній склянці?",
        "Переведи 3 столові ложки в мілілітри.",
        "Чим замінити яйця у випічці?",
        "Чим замінити молоко?",
        "Розрахуй час приготування борщу на 12 порцій.",
        "Запам'ятай, що я не їм глютен.",
        "Знайди мені рецепт без глютену.",
        "Чим замінити шоколад?"
    ]

    messages = []

    print("\n" + "=" * 60)
    print("🧪 АВТОМАТИЧНЕ ТЕСТУВАННЯ CHEFBOT")
    print("=" * 60)

    for i, query in enumerate(test_queries, start=1):
        print(f"\nТест #{i}")
        print("-" * 60)
        print(f"👤 Запит користувача: {query}")

        messages.append({"role": "user", "content": query})

        try:
            result_state = agent.invoke({"messages": messages})

            messages = result_state["messages"]
            last_message = messages[-1]

            print("✅ Відповідь ChefBot:")

            if isinstance(last_message, dict):
                print(last_message.get("content", ""))
            else:
                print(last_message.content)

        except Exception as e:
            print("⚠️ Помилка під час тесту.")
            print(f"Причина: {e}")

        print("-" * 60)

    print("\n✅ Автоматичне тестування завершено.")


run_automatic_tests()


🧪 АВТОМАТИЧНЕ ТЕСТУВАННЯ CHEFBOT

Тест #1
------------------------------------------------------------
👤 Запит користувача: Що можна приготувати з курки та картоплі?
⚠️ Помилка під час тесту.
Причина: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 41.85